In [1]:
%%capture
!pip install --quiet ultralytics==8.3.27 torch>=2.2.0 torchvision>=0.17.0 numpy==1.26.4 opencv-python>=4.8.0 scipy>=1.10.0 filterpy>=1.4.5 scikit-learn>=1.3.0 tqdm>=4.66.0 pyyaml>=6.0.1 matplotlib>=3.8.0

In [2]:
%%writefile config.py

from dataclasses import dataclass

@dataclass
class PoseConfig:
    # YOLOv8-Pose weights (downloaded automatically by ultralytics if missing)
    weights: str = "yolov8m-pose.pt"
    imgsz: int = 640
    conf: float = 0.25
    iou: float = 0.5
    device: str = "cuda"  # "cpu" or "cuda" 
    max_det: int = 50

@dataclass
class TrackConfig:
    max_age: int = 30        # frames to keep lost track
    min_hits: int = 5       # confirmations before reporting a track
    iou_threshold: float = 0.0  # Now works correctly with fixed logic

@dataclass
class FeatureConfig:
    seq_len: int = 30        # frames per clip window (1-2s @ 15-30fps)
    stride: int = 15         # overlap stride between clips
    kp_count: int = 17       # COCO-style keypoints
    smooth_sigma: float = 0.0  # no gaussian smoothing by default

@dataclass
class TrainConfig:
    batch_size: int = 16
    lr: float = 1e-3
    max_epochs: int = 10
    hidden_size: int = 128
    num_layers: int = 1
    dropout: float = 0.2
    num_classes: int = 2
    workers: int = 0
    # Device configs - separate for pose vs model training
    pose_device: str = "cuda"      # YOLO-Pose (CPU for compatibility)
    model_device: str = "cuda"    # BiLSTM model (auto-detect best available)

@dataclass
class RuntimeConfig:
    display: bool = True
    alert_threshold: float = 0.8
    save_alert_clips: bool = True
    out_dir: str = "alerts"
    


Writing config.py


In [3]:
%%writefile step2_pose.py
import torch
import numpy as np
from typing import Iterable, List
from ultralytics import YOLO


class PoseEstimator:
    def __init__(self, weights="yolov8n-pose.pt", imgsz=640, conf=0.25, iou=0.5, device="auto", max_det=50):
        self.model = YOLO(weights)
        self.imgsz = imgsz
        self.conf = conf
        self.iou = iou
        self.device = device
        self.max_det = max_det

    def _result_to_detections(self, res) -> List[tuple]:
        detections = []
        if res.boxes is None or res.keypoints is None:
            return detections
        boxes = res.boxes.xyxy.cpu().numpy()
        scores = res.boxes.conf.cpu().numpy()
        classes = res.boxes.cls.cpu().numpy()
        keypoints = res.keypoints.data.cpu().numpy()
        for bbox, score, cls, kp in zip(boxes, scores, classes, keypoints):
            if int(cls) != 0:  # keep only 'person' class
                continue
            detections.append(
                (bbox.astype(float), float(score), int(cls), kp[:, :3].astype(float))
            )
        return detections

    @torch.inference_mode()
    def infer(self, frame):
        # returns list of detections: [ (xyxy, score, cls, keypoints(17x3)) , ...]
        res = self.model.predict(
            source=frame,
            imgsz=self.imgsz,
            conf=self.conf,
            iou=self.iou,
            device=self.device,
            max_det=self.max_det,
            verbose=False,
        )[0]
        return self._result_to_detections(res)

    @torch.inference_mode()
    def infer_batch(self, frames: Iterable[np.ndarray]) -> List[List[tuple]]:
        frames = list(frames)
        if len(frames) == 0:
            return []
        results = self.model.predict(
            source=frames,
            imgsz=self.imgsz,
            conf=self.conf,
            iou=self.iou,
            device=self.device,
            max_det=self.max_det,
            verbose=False,
        )
        return [self._result_to_detections(res) for res in results]


Writing step2_pose.py


In [4]:
%%writefile tracking.py
import numpy as np
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter + 1e-6
    return inter / union

class KFTrack:
    count = 0
    def __init__(self, bbox, keypoints):
        self.id = KFTrack.count; KFTrack.count += 1
        self.kf = KalmanFilter(dim_x=8, dim_z=4)
        dt = 1.0
        # state: [x, y, w, h, vx, vy, vw, vh]
        self.kf.F = np.eye(8)
        for i in range(4):
            self.kf.F[i, i+4] = dt
        self.kf.H = np.zeros((4,8))
        self.kf.H[0,0] = self.kf.H[1,1] = self.kf.H[2,2] = self.kf.H[3,3] = 1.0
        self.kf.P *= 10.0
        self.kf.R *= 1.0
        self.kf.Q *= 0.01
        self.time_since_update = 0
        self.hits = 1
        self.keypoints = keypoints  # last kp
        x, y, w, h = self._xyxy_to_xywh(bbox)
        self.kf.x[:4] = np.array([[x],[y],[w],[h]])

    def predict(self):
        self.kf.predict()
        self.time_since_update += 1
        return self.get_bbox()

    def update(self, bbox, keypoints):
        x, y, w, h = self._xyxy_to_xywh(bbox)
        self.kf.update(np.array([x,y,w,h]))
        self.time_since_update = 0
        self.hits += 1
        self.keypoints = keypoints

    def get_bbox(self):
        x,y,w,h = self.kf.x[:4].flatten()
        return self._xywh_to_xyxy([x,y,w,h])

    @staticmethod
    def _xyxy_to_xywh(b):
        x = (b[0]+b[2])/2.0
        y = (b[1]+b[3])/2.0
        w = (b[2]-b[0])
        h = (b[3]-b[1])
        return x,y,w,h

    @staticmethod
    def _xywh_to_xyxy(b):
        x,y,w,h = b
        return np.array([x-w/2, y-h/2, x+w/2, y+h/2])

class Tracker:
    def __init__(self, max_age=30, min_hits=2, iou_threshold=0.3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.tracks = []

    def step(self, detections):
        # detections: list of (bbox, score, cls, kps(17x3))
        # Predict existing
        for t in self.tracks:
            t.predict()

        if len(detections)==0 and len(self.tracks)==0:
            return []

        # Match
        cost = np.ones((len(self.tracks), len(detections))) * 1.0
        for i,t in enumerate(self.tracks):
            tb = t.get_bbox()
            for j,d in enumerate(detections):
                db = d[0]
                cost[i,j] = 1 - iou(tb, db)  # Cost = 1 - IoU (lower is better)
        if len(self.tracks)>0 and len(detections)>0:
            r,c = linear_sum_assignment(cost)
            assigned = set()
            used_tracks = set()
            for i,j in zip(r,c):
                if cost[i,j] <= (1 - self.iou_threshold):  # Fixed: IoU >= threshold
                    self.tracks[i].update(detections[j][0], detections[j][3])
                    assigned.add(j); used_tracks.add(i)
            # new tracks for unassigned detections
            for j,d in enumerate(detections):
                if j not in assigned:
                    self.tracks.append(KFTrack(d[0], d[3]))
            # prune old
            survivors = []
            for idx,t in enumerate(self.tracks):
                if t.time_since_update <= self.max_age:
                    survivors.append(t)
            self.tracks = survivors
        else:
            # if no tracks or no detections: spawn new from detections
            for d in detections:
                self.tracks.append(KFTrack(d[0], d[3]))

        # Output confirmed tracks
        outputs = []
        for t in self.tracks:
            if t.hits >= self.min_hits:
                outputs.append((t.id, t.get_bbox(), t.keypoints))
        return outputs


Writing tracking.py


In [5]:
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
from tqdm import tqdm

from config import PoseConfig, FeatureConfig, TrackConfig
from step2_pose import PoseEstimator
from tracking import Tracker

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
def read_video_frames(video_path: Path) -> List[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    frames: List[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()
    return frames

In [7]:
def generate_windows(num_frames: int, seq_len: int, stride: int) -> List[Tuple[int, int]]:
    if num_frames == 0:
        return []
    if num_frames <= seq_len:
        return [(0, num_frames)]
    windows: List[Tuple[int, int]] = []
    for start in range(0, num_frames - seq_len + 1, stride):
        end = start + seq_len
        windows.append((start, end))
    if not windows:
        windows.append((num_frames - seq_len, num_frames))
    return windows

In [8]:
CONF_THRESH = 0.2     # joints below this confidence get zeroed
MIN_SCALE = 1e-2      # prevents division by tiny shoulder distances
MAX_RADIUS = 2.0      # drop joints > 2 body lengths away after normalization

def window_skeleton(
    frames: list[np.ndarray],
    pose: PoseEstimator,
    track_cfg: TrackConfig,
    seq_len: int,
    max_persons: int = 1,
) -> np.ndarray:
    tracker = Tracker(
        max_age=track_cfg.max_age,
        min_hits=track_cfg.min_hits,
        iou_threshold=track_cfg.iou_threshold,
    )

    keypoints_by_track: dict[int, list[np.ndarray]] = {}
    detections_per_frame = pose.infer_batch(frames)
    for detections in detections_per_frame:
        detections = sorted(detections, key=lambda d: d[1], reverse=True)[:max_persons]
        tracks = tracker.step(detections)
        for track_id, _, keypoints in tracks:
            keypoints_by_track.setdefault(track_id, []).append(keypoints)

    if not keypoints_by_track:
        return np.zeros((seq_len, 17, 7), dtype=np.float32)

    def track_score(seq):
        conf = [kp[:, 2].mean() for kp in seq]
        return len(seq), float(np.mean(conf))

    best_track = max(keypoints_by_track.values(), key=track_score)

    if len(best_track) >= seq_len:
        best_track = best_track[:seq_len]
    else:
        pad = [np.zeros_like(best_track[0]) for _ in range(seq_len - len(best_track))]
        best_track = best_track + pad

    skeleton = np.stack(best_track, axis=0).astype(np.float32)  # (T, 17, 3)

    H, W = frames[0].shape[:2]
    skeleton[..., 0] /= W
    skeleton[..., 1] /= H

    coords = skeleton[..., :2]
    conf = skeleton[..., 2:3]

    # zero unreliable joints before any differencing
    low_conf = conf < CONF_THRESH
    low_conf_mask = np.repeat(low_conf, 2, axis=2)  # (T, V, 2)
    coords[low_conf_mask] = 0.0

    hip_center = coords[:, [11, 12], :].mean(axis=1, keepdims=True)
    coords -= hip_center

    shoulder = coords[:, [5, 6], :]
    per_frame_len = np.linalg.norm(shoulder[:, 0] - shoulder[:, 1], axis=-1)
    valid = per_frame_len > MIN_SCALE
    scale = float(per_frame_len[valid].mean()) if np.any(valid) else 1.0
    coords /= max(scale, MIN_SCALE)

    # spatial sanity: drop joints drifting too far from torso
    
    radius_mask = np.linalg.norm(coords, axis=-1, keepdims=True) > MAX_RADIUS
    radius_mask = np.repeat(radius_mask, 2, axis=2)
    coords[radius_mask] = 0.0

    vel = np.zeros_like(coords)
    vel[1:] = coords[1:] - coords[:-1]
    acc = np.zeros_like(coords)
    acc[2:] = vel[2:] - vel[1:-1]

    skeleton_aug = np.concatenate([coords, vel, acc, conf], axis=-1)  # (T, 17, 7)
    return skeleton_aug





In [9]:
def process_video(
    video_path: Path,
    label: int,
    pose: PoseEstimator,
    feature_cfg: FeatureConfig,
    track_cfg: TrackConfig,
    max_persons: int,
) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    frames = read_video_frames(video_path)
    windows = generate_windows(len(frames), feature_cfg.seq_len, feature_cfg.stride)
    skeletons: list[np.ndarray] = []
    for start, end in windows:
        window_frames = frames[start:end]
        skeletons.append(
            window_skeleton(
                frames=window_frames,
                pose=pose,
                track_cfg=track_cfg,
                seq_len=feature_cfg.seq_len,
                max_persons=max_persons,
            )
        )
    if not skeletons:
        skeletons.append(np.zeros((feature_cfg.seq_len, 17, 3), dtype=np.float32))

    skeleton_array = np.stack(skeletons, axis=0)  # (num_windows, T, 17, 3)
    labels = np.full((skeleton_array.shape[0],), label, dtype=np.int64)
    stats = {
        "num_frames": len(frames),
        "num_windows": skeleton_array.shape[0],
    }
    return skeleton_array, labels, stats

In [10]:
import torch
from pathlib import Path

print(f"CUDA available: {torch.cuda.is_available()} | device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Using torch device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Enable a GPU accelerator in Kaggle settings.")

# Update these paths to match your Kaggle dataset layout
DATA_ROOT = Path('/kaggle/input/rwf2000/RWF-2000')  # TODO: set to your dataset path
OUTPUT_ROOT = Path('/kaggle/working/precomputed_skeletons')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SPLITS = ['train', 'val']  # adjust if you have a different split naming
NUM_WORKERS = 0            # GPU inference runs sequentially; increase cautiously if memory allows
MAX_PERSONS = 10           # top-N persons per frame

print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Splits: {SPLITS}")


CUDA available: True | device count: 1
Using torch device: Tesla P100-PCIE-16GB
Data root: /kaggle/input/rwf2000/RWF-2000
Output root: /kaggle/working/precomputed_skeletons
Splits: ['train', 'val']


In [11]:
from pathlib import Path
import numpy as np
from tqdm import tqdm

pose_cfg = PoseConfig()
track_cfg = TrackConfig()
feature_cfg = FeatureConfig()
pose = PoseEstimator(
    weights=pose_cfg.weights,
    imgsz=pose_cfg.imgsz,
    conf=pose_cfg.conf,
    iou=pose_cfg.iou,
    device=pose_cfg.device,
    max_det=pose_cfg.max_det,
)

label_pairs = [("Fight", 1), ("NonFight", 0)]


manifest = []
for split in SPLITS:
    for label_name, label in label_pairs:
        src_dir = DATA_ROOT / split / label_name
        if not src_dir.exists():
            continue
        dst_dir = OUTPUT_ROOT / split / label_name
        dst_dir.mkdir(parents=True, exist_ok=True)

        videos = sorted(list(src_dir.glob("*.mp4")) + list(src_dir.glob("*.avi")))
        
        # videos = videos[:50]
        
        for video_path in tqdm(videos, desc=f"{split}/{label_name}", unit="video"):
            out_path = dst_dir / f"{video_path.stem}.npz"
            skeletons, labels, stats = process_video(
                video_path=video_path,
                label=label,
                pose=pose,
                feature_cfg=feature_cfg,
                track_cfg=track_cfg,
                max_persons=1,  # keep top track; adjust if needed
            )
            np.savez_compressed(out_path, skeletons=skeletons, labels=labels)
            manifest.append({
                "split": split,
                "label_name": label_name,
                "label": label,
                "video": video_path.name,
                "num_frames": stats["num_frames"],
                "num_windows": stats["num_windows"],
                "output": str(out_path.relative_to(OUTPUT_ROOT)),
            })

# Optional: write metadata
import json
(OUTPUT_ROOT / "manifest.json").write_text(json.dumps({
    "pose_cfg": asdict(pose_cfg),
    "feature_cfg": asdict(feature_cfg),
    "track_cfg": asdict(track_cfg),
    "manifest": manifest,
}, indent=2))
print("Done:", len(manifest), "videos")


100%|██████████| 50.8M/50.8M [00:00<00:00, 147MB/s]
val/NonFight: 100%|██████████| 200/200 [09:13<00:00,  2.77s/video]

Done: 2000 videos


In [13]:
import shutil

# Path to the folder you want to zip
folder_path = '/kaggle/working/precomputed_skeletons'
# Output zip file path
zip_path = '/kaggle/working/precomputed_skeletons2.zip'

# Create zip archive from the folder
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', folder_path)


'/kaggle/working/precomputed_skeletons2.zip'